In [1]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Imports (your existing setup)
# ════════════════════════════════════════════════════════════════
from langchain_chroma import Chroma
from pdf_chunk import text_spliter
from models import get_model, get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# langchain-classic — correct imports for LangChain 1.2.15
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain_classic.retrievers.document_compressors.chain_filter import LLMChainFilter
from langchain_classic.retrievers.document_compressors.embeddings_filter import EmbeddingsFilter
from langchain_classic.retrievers.document_compressors.base import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Base setup (same as your file)
# ════════════════════════════════════════════════════════════════
llm = get_model()
embedding_model = get_embeddings()
chunks = text_spliter

chroma_vectorstore = Chroma.from_documents(chunks, embedding_model)

chroma_retriever = chroma_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "include_metadata": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2567.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Helper
# ════════════════════════════════════════════════════════════════
def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join(
        [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
    ))

def format_docs(docs):
    """Convert list of Documents to a single context string for the LLM."""
    return "\n\n".join([d.page_content for d in docs])

In [4]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Prompt (shared across all chains)
# ════════════════════════════════════════════════════════════════
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

In [5]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Chain A: LLMChainExtractor
#           LLM surgically extracts relevant sentences per chunk
# ════════════════════════════════════════════════════════════════
compressor_extractor = LLMChainExtractor.from_llm(llm)

compression_retriever_A = ContextualCompressionRetriever(
    base_compressor=compressor_extractor,
    base_retriever=chroma_retriever
)

chain_A = (
    {
        "context": compression_retriever_A | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

query = "what is current university of rohanta"
print("═" * 60)
print("Chain A — LLMChainExtractor")
print("═" * 60)
print(chain_A.invoke(query))

════════════════════════════════════════════════════════════
Chain A — LLMChainExtractor
════════════════════════════════════════════════════════════
Savitribai Phule Pune University, Pune, India


In [6]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — Chain B: LLMChainFilter
#           LLM gives binary YES/NO — keeps or drops full chunks
# ════════════════════════════════════════════════════════════════
compressor_filter = LLMChainFilter.from_llm(llm)

compression_retriever_B = ContextualCompressionRetriever(
    base_compressor=compressor_filter,
    base_retriever=chroma_retriever
)

chain_B = (
    {
        "context": compression_retriever_B | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("═" * 60)
print("Chain B — LLMChainFilter")
print("═" * 60)
print(chain_B.invoke(query))

════════════════════════════════════════════════════════════
Chain B — LLMChainFilter
════════════════════════════════════════════════════════════
I don't know.


In [7]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Chain C: EmbeddingsFilter
#           No LLM — cosine similarity drop below threshold
# ════════════════════════════════════════════════════════════════
embeddings_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.40  # lower threshold since MiniLM scores tend to be lower
)

compression_retriever_C = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=chroma_retriever
)

chain_C = (
    {
        "context": compression_retriever_C | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("═" * 60)
print("Chain C — EmbeddingsFilter (no LLM, fastest)")
print("═" * 60)
print(chain_C.invoke(query))

════════════════════════════════════════════════════════════
Chain C — EmbeddingsFilter (no LLM, fastest)
════════════════════════════════════════════════════════════
I don't know.


In [8]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — Chain D: DocumentCompressorPipeline
#           re-split → deduplicate → relevance filter (all-in-one)
# ════════════════════════════════════════════════════════════════
splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separator=". "
)

redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding_model)

relevant_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.40
)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter]
)

compression_retriever_D = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=chroma_retriever
)

chain_D = (
    {
        "context": compression_retriever_D | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("═" * 60)
print("Chain D — Pipeline (splitter + redundant + relevance filter)")
print("═" * 60)
print(chain_D.invoke(query))

════════════════════════════════════════════════════════════
Chain D — Pipeline (splitter + redundant + relevance filter)
════════════════════════════════════════════════════════════
I don't know.


In [9]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Compression ratio comparison (all 4 methods)
# ════════════════════════════════════════════════════════════════
test_query = "what is current university of rohanta"

raw_docs     = chroma_retriever.invoke(test_query)
comp_A_docs  = compression_retriever_A.invoke(test_query)
comp_B_docs  = compression_retriever_B.invoke(test_query)
comp_C_docs  = compression_retriever_C.invoke(test_query)
comp_D_docs  = compression_retriever_D.invoke(test_query)

def char_len(docs):
    return len("\n\n".join([d.page_content for d in docs]))

original_len = char_len(raw_docs)

print(f"{'Method':<30} {'Docs':>5} {'Chars':>8} {'Ratio':>8}")
print("-" * 55)
print(f"{'Raw (no compression)':<30} {len(raw_docs):>5} {original_len:>8} {'1.00x':>8}")
for label, docs_list in [
    ("A — LLMChainExtractor",   comp_A_docs),
    ("B — LLMChainFilter",      comp_B_docs),
    ("C — EmbeddingsFilter",    comp_C_docs),
    ("D — Pipeline compressor", comp_D_docs),
]:
    cl = char_len(docs_list)
    ratio = original_len / (cl + 1e-5)
    print(f"{label:<30} {len(docs_list):>5} {cl:>8} {ratio:>7.2f}x")

Method                          Docs    Chars    Ratio
-------------------------------------------------------
Raw (no compression)              10     2753    1.00x
A — LLMChainExtractor             10     1956    1.41x
B — LLMChainFilter                 3     1246    2.21x
C — EmbeddingsFilter               1      495    5.56x
D — Pipeline compressor            1      283    9.73x
